# Market-Calibrated Option Pricing and Volatility Surface

## Research question

**How closely do Black-Scholes and binomial prices match historical SPY option-market prices, and how do errors vary by moneyness, maturity, volatility, and liquidity?**

This notebook uses the supplied SPY_options.parquet file. It cleans the chain, uses the mid price as the market reference, derives implied volatility, and evaluates model prices on a representative sample. Install missing packages in the active kernel with: pip install pyarrow plotly ipywidgets.

## Conventions

- Market price = midpoint of non-negative bid and ask; rows with invalid quotes are removed.
- Time to maturity is measured in years using 365.25 days.
- Moneyness is spot divided by strike.
- Black-Scholes uses the option-implied volatility, allowing the analysis to isolate the effect of its distributional assumptions.
- The binomial model uses a Cox-Ross-Rubinstein tree and supports American exercise.
- Pricing error is model price minus market mid price.

Because option chains can be very large, the expensive implied-volatility and tree calculations operate on a reproducible sample. Increase MAX_ANALYSIS_ROWS after confirming runtime.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

RISK_FREE_RATE = 0.045       # Annual continuously compounded proxy; replace with quote-date Treasury rates when available.
DIVIDEND_YIELD = 0.013       # SPY annual dividend-yield proxy.
TREE_STEPS = 150
MAX_ANALYSIS_ROWS = 8_000
RANDOM_SEED = 42
pd.options.display.float_format = '{:,.4f}'.format

data_candidates = [Path('SPY_options.parquet'), Path('../SPY_options.parquet')]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Could not find SPY_options.parquet. Run with the Q_Fin folder as the working directory.')
print(f'Using option data: {DATA_PATH.resolve()}')

Using option data: C:\Users\Samarth Sitape\OneDrive\Notes\Q_Fin\SPY_options.parquet


In [3]:
raw_options = pd.read_parquet(DATA_PATH)
print(f'Raw rows: {len(raw_options):,}; columns: {list(raw_options.columns)}')

def resolve_column(frame, candidates, required=True):
    lookup = {str(column).lower().replace('_', '').replace(' ', ''): column for column in frame.columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    if required:
        raise KeyError(f'Missing a required field. Tried {candidates}; available columns are {list(frame.columns)}')
    return None

# Accept common vendor naming conventions; edit candidate lists once if your file uses a custom schema.
column_map = {
    'date': resolve_column(raw_options, ['quotedate', 'date', 'trade date', 'timestamp']),
    'expiry': resolve_column(raw_options, ['expiration', 'expiry', 'expirationdate', 'exdate']),
    'strike': resolve_column(raw_options, ['strike', 'strikeprice']),
    'type': resolve_column(raw_options, ['optiontype', 'type', 'callput', 'right']),
    'spot': resolve_column(raw_options, ['underlyingprice', 'underlyinglast', 'spot', 'stockprice', 'underlying'], required=False),
    'bid': resolve_column(raw_options, ['bid', 'bestbid']),
    'ask': resolve_column(raw_options, ['ask', 'bestask']),
    'volume': resolve_column(raw_options, ['volume', 'tradevolume'], required=False),
    'open_interest': resolve_column(raw_options, ['openinterest', 'oi'], required=False),
    'iv_vendor': resolve_column(raw_options, ['impliedvolatility', 'iv', 'impliedvol'], required=False),
}

options = pd.DataFrame({name: raw_options[column] if column is not None else np.nan for name, column in column_map.items()})
options['date'] = pd.to_datetime(options['date'])
options['expiry'] = pd.to_datetime(options['expiry'])
for column in ['strike', 'spot', 'bid', 'ask', 'volume', 'open_interest', 'iv_vendor']:
    options[column] = pd.to_numeric(options[column], errors='coerce')
# The supplied file does not include SPY spot. Merge unadjusted SPY closes by quote date when needed.
if options['spot'].isna().any():
    spot_history = yf.download('SPY', start=options['date'].min() - pd.Timedelta(days=7), end=options['date'].max() + pd.Timedelta(days=7), auto_adjust=False, progress=False)['Close']
    if isinstance(spot_history, pd.DataFrame):
        spot_history = spot_history.iloc[:, 0]
    spot_history.index = pd.to_datetime(spot_history.index).normalize()
    options['spot'] = options['spot'].fillna(options['date'].dt.normalize().map(spot_history))
options['type'] = options['type'].astype(str).str.upper().str[0]
options['market_price'] = (options['bid'] + options['ask']) / 2
options['spread_pct'] = (options['ask'] - options['bid']) / options['market_price']
options['ttm'] = (options['expiry'] - options['date']).dt.total_seconds() / (365.25 * 24 * 3600)
options['moneyness'] = options['spot'] / options['strike']
options = options.query('spot > 0 and strike > 0 and ttm > 0 and bid >= 0 and ask >= bid and market_price > 0')
options = options.loc[options['type'].isin(['C', 'P'])].copy()
options = options.loc[options['spread_pct'].between(0, 2)].copy()
analysis_options = options.sample(min(MAX_ANALYSIS_ROWS, len(options)), random_state=RANDOM_SEED).copy()
print(f'Valid quoted options: {len(options):,}; pricing sample: {len(analysis_options):,}')

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
def black_scholes(spot, strike, ttm, rate, sigma, option_type, dividend=0.0):
    if min(spot, strike, ttm, sigma) <= 0:
        return np.nan
    d1 = (np.log(spot / strike) + (rate - dividend + 0.5 * sigma**2) * ttm) / (sigma * np.sqrt(ttm))
    d2 = d1 - sigma * np.sqrt(ttm)
    if option_type == 'C':
        return spot * np.exp(-dividend * ttm) * norm.cdf(d1) - strike * np.exp(-rate * ttm) * norm.cdf(d2)
    return strike * np.exp(-rate * ttm) * norm.cdf(-d2) - spot * np.exp(-dividend * ttm) * norm.cdf(-d1)


def implied_volatility(market_price, spot, strike, ttm, rate, option_type, dividend=0.0):
    objective = lambda sigma: black_scholes(spot, strike, ttm, rate, sigma, option_type, dividend) - market_price
    try:
        return brentq(objective, 1e-4, 5.0, maxiter=100)
    except ValueError:
        return np.nan


def crr_binomial(spot, strike, ttm, rate, sigma, option_type, steps=TREE_STEPS, dividend=0.0):
    dt = ttm / steps
    up, down = np.exp(sigma * np.sqrt(dt)), np.exp(-sigma * np.sqrt(dt))
    probability = (np.exp((rate - dividend) * dt) - down) / (up - down)
    if not 0 < probability < 1:
        return np.nan
    terminal_spot = spot * up ** np.arange(steps, -1, -1) * down ** np.arange(0, steps + 1)
    values = np.maximum(terminal_spot - strike, 0) if option_type == 'C' else np.maximum(strike - terminal_spot, 0)
    discount = np.exp(-rate * dt)
    for step in range(steps - 1, -1, -1):
        values = discount * (probability * values[:-1] + (1 - probability) * values[1:])
        current_spot = spot * up ** np.arange(step, -1, -1) * down ** np.arange(0, step + 1)
        intrinsic = np.maximum(current_spot - strike, 0) if option_type == 'C' else np.maximum(strike - current_spot, 0)
        values = np.maximum(values, intrinsic)
    return values[0]


def bs_greeks(spot, strike, ttm, rate, sigma, option_type, dividend=0.0):
    d1 = (np.log(spot / strike) + (rate - dividend + 0.5 * sigma**2) * ttm) / (sigma * np.sqrt(ttm))
    d2 = d1 - sigma * np.sqrt(ttm)
    sign = 1 if option_type == 'C' else -1
    delta = sign * np.exp(-dividend * ttm) * norm.cdf(sign * d1)
    gamma = np.exp(-dividend * ttm) * norm.pdf(d1) / (spot * sigma * np.sqrt(ttm))
    vega = spot * np.exp(-dividend * ttm) * norm.pdf(d1) * np.sqrt(ttm) / 100
    theta = (-(spot * np.exp(-dividend * ttm) * norm.pdf(d1) * sigma) / (2 * np.sqrt(ttm))
             - sign * rate * strike * np.exp(-rate * ttm) * norm.cdf(sign * d2)
             + sign * dividend * spot * np.exp(-dividend * ttm) * norm.cdf(sign * d1)) / 365
    rho = sign * strike * ttm * np.exp(-rate * ttm) * norm.cdf(sign * d2) / 100
    return pd.Series({'Delta': delta, 'Gamma': gamma, 'Vega per 1 vol point': vega, 'Theta per day': theta, 'Rho per 1 rate point': rho})


analysis_options['implied_vol'] = analysis_options.apply(lambda row: implied_volatility(row.market_price, row.spot, row.strike, row.ttm, RISK_FREE_RATE, row.type, DIVIDEND_YIELD), axis=1)
analysis_options = analysis_options.query('implied_vol > 0 and implied_vol < 5').copy()
analysis_options['bs_price'] = analysis_options.apply(lambda row: black_scholes(row.spot, row.strike, row.ttm, RISK_FREE_RATE, row.implied_vol, row.type, DIVIDEND_YIELD), axis=1)
analysis_options['binomial_price'] = analysis_options.apply(lambda row: crr_binomial(row.spot, row.strike, row.ttm, RISK_FREE_RATE, row.implied_vol, row.type, dividend=DIVIDEND_YIELD), axis=1)
analysis_options['bs_error'] = analysis_options.bs_price - analysis_options.market_price
analysis_options['binomial_error'] = analysis_options.binomial_price - analysis_options.market_price
analysis_options[['moneyness', 'ttm', 'implied_vol', 'market_price', 'bs_price', 'binomial_price', 'bs_error', 'binomial_error']].describe()

In [ ]:
# Volatility smile: show median IV in rounded maturity buckets so that noisy individual quotes do not dominate.
analysis_options['maturity_bucket'] = pd.cut(analysis_options.ttm, [0, 1/12, .25, .5, 1, 2, np.inf], labels=['<1m', '1-3m', '3-6m', '6-12m', '1-2y', '>2y'])
analysis_options['moneyness_bucket'] = pd.cut(analysis_options.moneyness, np.arange(.60, 1.46, .05))
smile = analysis_options.groupby(['maturity_bucket', 'moneyness_bucket'], observed=True).implied_vol.median().reset_index()
fig = px.line(smile, x='moneyness_bucket', y='implied_vol', color='maturity_bucket', markers=True, title='SPY implied-volatility smile by maturity')
fig.update_yaxes(tickformat='.0%', title='Implied volatility')
fig.update_xaxes(title='Spot / strike moneyness bucket')
fig.show()

surface = analysis_options.assign(moneyness_round=analysis_options.moneyness.round(2), ttm_days=(analysis_options.ttm * 365.25).round())
surface = surface.groupby(['ttm_days', 'moneyness_round']).implied_vol.median().reset_index()
surface_grid = surface.pivot(index='ttm_days', columns='moneyness_round', values='implied_vol').sort_index().sort_index(axis=1)
fig = go.Figure(data=[go.Surface(x=surface_grid.columns, y=surface_grid.index, z=surface_grid.values, colorscale='Viridis')])
fig.update_layout(title='Interactive implied-volatility surface', scene=dict(xaxis_title='Moneyness', yaxis_title='Days to expiry', zaxis_title='Implied volatility'))
fig.show()

analysis_options['liquidity_bucket'] = pd.qcut(analysis_options.spread_pct.rank(method='first'), 4, labels=['Tightest', 'Q2', 'Q3', 'Widest'])
error_table = analysis_options.groupby(['maturity_bucket', 'moneyness_bucket', 'liquidity_bucket'], observed=True).agg(
    Contracts=('market_price', 'size'), BS_MAE=('bs_error', lambda x: x.abs().mean()),
    Binomial_MAE=('binomial_error', lambda x: x.abs().mean()), Mean_IV=('implied_vol', 'mean')
).reset_index()
display(error_table.sort_values('BS_MAE', ascending=False).head(20))

heat = analysis_options.groupby(['maturity_bucket', 'moneyness_bucket'], observed=True).bs_error.apply(lambda x: x.abs().mean()).unstack()
plt.figure(figsize=(12, 5)); plt.imshow(heat, aspect='auto', cmap='magma'); plt.colorbar(label='Black-Scholes mean absolute error');
plt.xticks(range(len(heat.columns)), heat.columns, rotation=45); plt.yticks(range(len(heat.index)), heat.index); plt.title('Pricing error by maturity and moneyness'); plt.tight_layout(); plt.show()

In [ ]:
# Interactive scenario pricer. Greeks use the same Black-Scholes assumptions as the displayed price.
@interact(spot=FloatSlider(value=600, min=300, max=900, step=5, description='Spot'),
          strike=FloatSlider(value=600, min=300, max=900, step=5, description='Strike'),
          days=IntSlider(value=90, min=7, max=730, step=7, description='Days'),
          volatility=FloatSlider(value=.20, min=.05, max=1.0, step=.01, description='Volatility'),
          option_type=Dropdown(options=['C', 'P'], value='C', description='Type'))
def interactive_pricer(spot, strike, days, volatility, option_type):
    ttm = days / 365.25
    bs_price = black_scholes(spot, strike, ttm, RISK_FREE_RATE, volatility, option_type, DIVIDEND_YIELD)
    tree_price = crr_binomial(spot, strike, ttm, RISK_FREE_RATE, volatility, option_type, dividend=DIVIDEND_YIELD)
    print(f'Black-Scholes price: {bs_price:.2f} | Binomial price: {tree_price:.2f}')
    display(bs_greeks(spot, strike, ttm, RISK_FREE_RATE, volatility, option_type, DIVIDEND_YIELD).to_frame('Value'))

## Interpretation

Use the grouped error table to identify where each model is least reliable. Persistent errors for deep out-of-the-money options, short maturities, or wide-spread contracts may reflect volatility-surface effects, discrete dividends, early exercise, stale quotes, or liquidity rather than a pure model defect. Since each model is fed market-implied volatility, errors should be interpreted as residual pricing differences after the market volatility level has been incorporated.